# Diccionarios · Preguntas y variantes

Claves repetidas y no hashables, claves inexistentes, modificar un diccionario mientras se recorre, orden de inserción, alias y copias, vistas y distintas formas de resolver el mismo problema.

## 1. Claves repetidas

¿Qué muestran `{'a': 1, 'b': 2, 'a': 3}` y `{1: 'uno', 1.0: 'uno coma cero', True: 'verdadero'}`?

Una clave repetida conserva la posición de su primera aparición y el valor de la última. `1`, `1.0` y `True` son iguales y tienen el mismo *hash*, así que son la misma clave; `'1'` y `1`, no.

In [1]:
print({"a": 1, "b": 2, "a": 3})
d = {1: "uno", 1.0: "uno coma cero", True: "verdadero"}
print(d, len(d), d[1.0], d[True])
print(hash(1) == hash(1.0) == hash(True), 1 == 1.0 == True)
print({0: "cero", False: "falso", 0.0: "cero real"}, {"1": "texto", 1: "número"})

{'a': 3, 'b': 2}
{1: 'verdadero'} 1 verdadero verdadero
True True
{0: 'cero real'} {'1': 'texto', 1: 'número'}


## 2. Claves no hashables

¿Cuáles de estas claves son válidas: `[1, 2]`, `(1, 2)`, `(1, [2])`, `{1, 2}`, `frozenset({1, 2})`, `{'a': 1}`, `'texto'`, `3.5` y `None`? Solo pueden ser claves los objetos *hashables* (en la práctica, los inmutables); una tupla lo es solo si todos sus elementos lo son. ¿Cómo se cuenta cuántas veces aparece cada ruta si las rutas son listas?

In [2]:
d = {}
for clave in [[1, 2], (1, 2), (1, [2]), {1, 2}, frozenset({1, 2}), {"a": 1}, "texto", 3.5, None]:
    try:
        d[clave] = True
        print(f"{clave!r:20} válida")
    except TypeError as e:
        print(f"{clave!r:20} TypeError: {e}")

[1, 2]               TypeError: unhashable type: 'list'
(1, 2)               válida
(1, [2])             TypeError: unhashable type: 'list'
{1, 2}               TypeError: unhashable type: 'set'
frozenset({1, 2})    válida
{'a': 1}             TypeError: unhashable type: 'dict'
'texto'              válida
3.5                  válida
None                 válida


In [3]:
visitas = {}
for ruta in (["casa", "trabajo"], ["casa", "gimnasio"], ["casa", "trabajo"]):
    clave = tuple(ruta)
    visitas[clave] = visitas.get(clave, 0) + 1
print(visitas)

{('casa', 'trabajo'): 2, ('casa', 'gimnasio'): 1}


## 3. Claves que no existen

Con `d = {'a': 1, 'b': 2}`, ¿qué dan `d['x']`, `d.get('x')`, `d.get('x', 0)`, `'x' in d`, `1 in d`, `1 in d.values()` y `('a', 1) in d.items()`? ¿Cambia `d` después de usar `get`?

`d[clave]` lanza `KeyError`; `get` devuelve `None` o el valor por defecto sin modificar el diccionario. `in` busca entre las claves, no entre los valores.

In [4]:
d = {"a": 1, "b": 2}
try:
    d["x"]
except KeyError as e:
    print("KeyError:", e)
print(d.get("x"), d.get("x", 0), "x" in d, 1 in d, 1 in d.values(), ("a", 1) in d.items())
print(d)

KeyError: 'x'
None 0 False False True True
{'a': 1, 'b': 2}


## 4. Modificar un diccionario mientras se recorre

Este programa quiere borrar los productos sin existencias. ¿Por qué falla y cómo queda el diccionario?

No se puede añadir ni borrar claves de un diccionario mientras se recorre: Python lo detecta y lanza `RuntimeError` (en este caso, tras haber borrado ya el primero). Corregirlo recorriendo una copia de las claves (`list(stock)`) o creando un diccionario nuevo con una comprensión. Cambiar solo los valores sí está permitido.

In [5]:
stock = {"pan": 0, "leche": 3, "huevos": 0, "café": 5}
try:
    for producto, cantidad in stock.items():
        if cantidad == 0:
            del stock[producto]
except RuntimeError as e:
    print("RuntimeError:", e)
print(stock)

d = {"a": 1}
try:
    for k in d:
        del d[k]
        d[k.upper()] = 1
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: dictionary changed size during iteration
{'leche': 3, 'huevos': 0, 'café': 5}
RuntimeError: dictionary keys changed during iteration


In [6]:
stock = {"pan": 0, "leche": 3, "huevos": 0, "café": 5}
for producto in list(stock):
    if stock[producto] == 0:
        del stock[producto]
print(stock)

stock = {"pan": 0, "leche": 3, "huevos": 0, "café": 5}
print({k: v for k, v in stock.items() if v > 0})
for k in stock:
    stock[k] += 10
print(stock)

{'leche': 3, 'café': 5}
{'leche': 3, 'café': 5}
{'pan': 10, 'leche': 13, 'huevos': 10, 'café': 15}


## 5. Orden de inserción

¿En qué orden se recorren las claves de un diccionario? ¿Qué pasa al reasignar una clave existente y al borrarla y volver a insertarla? ¿Son iguales dos diccionarios con las mismas parejas en distinto orden?

Desde Python 3.7 los diccionarios conservan el orden de inserción. Reasignar no cambia la posición; borrar y volver a insertar lleva la clave al final. La igualdad no tiene en cuenta el orden. Para ordenar un diccionario se crea uno nuevo con `sorted`.

In [7]:
d = {"c": 3, "a": 1, "b": 2}
print(list(d))
d["a"] = 10
print(list(d))
del d["c"]
d["c"] = 30
print(d)
print({"x": 1, "y": 2} == {"y": 2, "x": 1}, list({"x": 1, "y": 2}) == list({"y": 2, "x": 1}))
print(dict(sorted(d.items())), dict(sorted(d.items(), key=lambda kv: kv[1], reverse=True)))
print(next(iter(d)), next(reversed(d)))

['c', 'a', 'b']
['c', 'a', 'b']
{'a': 10, 'b': 2, 'c': 30}
True False
{'a': 10, 'b': 2, 'c': 30} {'c': 30, 'a': 10, 'b': 2}
a c


## 6. Alias, copia superficial y copia profunda

¿Qué muestra este programa?

`alias = a` no copia nada: es otro nombre para el mismo diccionario. `dict(a)`, `a.copy()` y `{**a}` crean un diccionario nuevo, pero sus valores son los mismos objetos (copia superficial), así que la lista de notas se comparte. `copy.deepcopy` copia también los valores anidados.

In [8]:
import copy

a = {"nombre": "Ana", "notas": [7, 8]}
alias = a
copias = {"alias": alias, "dict(a)": dict(a), "a.copy()": a.copy(), "{**a}": {**a}, "deepcopy": copy.deepcopy(a)}
a["nombre"] = "Eva"
a["notas"].append(10)
for nombre, d in copias.items():
    print(f"{nombre:9} {d}  ¿mismo dict? {d is a}  ¿misma lista? {d['notas'] is a['notas']}")

alias     {'nombre': 'Eva', 'notas': [7, 8, 10]}  ¿mismo dict? True  ¿misma lista? True
dict(a)   {'nombre': 'Ana', 'notas': [7, 8, 10]}  ¿mismo dict? False  ¿misma lista? True
a.copy()  {'nombre': 'Ana', 'notas': [7, 8, 10]}  ¿mismo dict? False  ¿misma lista? True
{**a}     {'nombre': 'Ana', 'notas': [7, 8, 10]}  ¿mismo dict? False  ¿misma lista? True
deepcopy  {'nombre': 'Ana', 'notas': [7, 8]}  ¿mismo dict? False  ¿misma lista? False


## 7. `dict.fromkeys` con un valor mutable

Se quiere preparar una lista vacía de tareas para cada día con `dict.fromkeys(dias, [])`. ¿Qué pasa al añadir una tarea al lunes?

`fromkeys` usa el mismo objeto como valor de todas las claves, así que todas comparten la misma lista. Con valores inmutables (`0`, `None`) no hay problema; con mutables hay que usar una comprensión o un `defaultdict(list)`.

In [9]:
from collections import defaultdict

dias = ["lun", "mar", "mié"]
tareas = dict.fromkeys(dias, [])
tareas["lun"].append("examen")
print(tareas, tareas["lun"] is tareas["mar"])

tareas = {d: [] for d in dias}
tareas["lun"].append("examen")
print(tareas)

tareas = defaultdict(list)
tareas["lun"].append("examen")
print(dict(tareas))
print(dict.fromkeys("abc", 0), dict.fromkeys(dias))

{'lun': ['examen'], 'mar': ['examen'], 'mié': ['examen']} True
{'lun': ['examen'], 'mar': [], 'mié': []}
{'lun': ['examen']}
{'a': 0, 'b': 0, 'c': 0} {'lun': None, 'mar': None, 'mié': None}


## 8. Vistas de un diccionario

¿Qué muestra este programa?

`keys()`, `values()` e `items()` no son listas sino *vistas*: reflejan los cambios posteriores del diccionario y no admiten índices. Las vistas de claves y de parejas se comportan como conjuntos (`&`, `-`, `|`, `==`); la de valores no, porque los valores pueden repetirse y no ser hashables.

In [10]:
precios = {"pan": 1.2, "leche": 0.95}
claves, valores = precios.keys(), precios.values()
precios["café"] = 3.4
print(claves, valores, len(claves))

otros = {"leche": 1.0, "té": 2.1}
print(sorted(precios.keys() & otros.keys()), sorted(precios.keys() - otros.keys()), sorted(precios.keys() | otros.keys()))
print(precios.items() & {("pan", 1.2), ("té", 2.1)}, claves == {"café", "leche", "pan"})
for prueba in (lambda: claves[0], lambda: valores & {1.2}):
    try:
        prueba()
    except TypeError as e:
        print("TypeError:", e)
print(list(claves)[0])

dict_keys(['pan', 'leche', 'café']) dict_values([1.2, 0.95, 3.4]) 3
['leche'] ['café', 'pan'] ['café', 'leche', 'pan', 'té']
{('pan', 1.2)} True
TypeError: 'dict_keys' object is not subscriptable
TypeError: unsupported operand type(s) for &: 'dict_values' and 'set'
pan


## 9. Funciones sobre un diccionario

Con `d = {'b': 3, 'a': 5, 'c': 1}`, ¿qué dan `len(d)`, `list(d)`, `sorted(d)`, `max(d)`, `max(d, key=d.get)`, `min(d.items(), key=lambda kv: kv[1])`, `sorted(d, key=d.get)`, `sum(d.values())`, `dict(zip(d.values(), d))` y `sum(d)`?

Las funciones que recorren un diccionario usan sus claves: `max(d)` es la clave mayor, no el valor mayor, y `sum(d)` intenta sumar las claves.

In [11]:
d = {"b": 3, "a": 5, "c": 1}
for expr in ("len(d)", "list(d)", "sorted(d)", "max(d)", "max(d, key=d.get)", "min(d.items(), key=lambda kv: kv[1])",
             "sorted(d, key=d.get)", "sum(d.values())", "dict(zip(d.values(), d))", "sum(d)"):
    try:
        print(f"{expr:38} {eval(expr)!r}")
    except TypeError as e:
        print(f"{expr:38} TypeError: {e}")

len(d)                                 3
list(d)                                ['b', 'a', 'c']
sorted(d)                              ['a', 'b', 'c']
max(d)                                 'c'
max(d, key=d.get)                      'a'
min(d.items(), key=lambda kv: kv[1])   ('c', 1)
sorted(d, key=d.get)                   ['c', 'b', 'a']
sum(d.values())                        9
dict(zip(d.values(), d))               {3: 'b', 5: 'a', 1: 'c'}
sum(d)                                 TypeError: unsupported operand type(s) for +: 'int' and 'str'


## 10. Buscar claves por valor

Escribir una función que devuelva la primera clave que tiene un valor dado (o `None`) y otra que devuelva todas. ¿Por qué los diccionarios no tienen un método `index(valor)`?

Un diccionario está pensado para buscar por clave en tiempo $O(1)$; buscar por valor obliga a recorrerlo entero, en $O(n)$. Si se hace a menudo, conviene construir el diccionario inverso una vez.

In [12]:
telefonos = {"Ana": "928 111 111", "Luis": "922 222 222", "Eva": "928 111 111"}

def primera_clave(d, valor):
    return next((k for k, v in d.items() if v == valor), None)

def todas_las_claves(d, valor):
    return [k for k, v in d.items() if v == valor]

print(primera_clave(telefonos, "928 111 111"), primera_clave(telefonos, "000"), todas_las_claves(telefonos, "928 111 111"))
print(hasattr(telefonos, "index"))

inverso = {}
for nombre, tlf in telefonos.items():
    inverso.setdefault(tlf, []).append(nombre)
assert all(inverso[v] == todas_las_claves(telefonos, v) for v in telefonos.values())
print(inverso)

Ana None ['Ana', 'Eva']
False
{'928 111 111': ['Ana', 'Eva'], '922 222 222': ['Luis']}


## 11. `del`, `pop`, `popitem` y `clear`

¿Qué muestran estas operaciones?

`pop` devuelve el valor borrado y admite un valor por defecto para no fallar; `del` y `pop` sin valor por defecto lanzan `KeyError` si la clave no existe. `popitem` quita la última pareja insertada (orden LIFO). `clear` vacía el diccionario y lo notan todos sus alias, mientras que `x = {}` solo hace que `x` apunte a otro diccionario.

In [13]:
d = {"a": 1, "b": 2, "c": 3, "d": 4}
print(d.pop("a"), d.pop("z", "no está"), d.popitem(), d)
del d["b"]
print(d)
for borra in (lambda: d.__delitem__("z"), lambda: d.pop("z"), lambda: {}.popitem()):
    try:
        borra()
    except KeyError as e:
        print("KeyError:", e)

x = {"k": 1}
alias = x
x = {}
print(alias)
x = alias
x.clear()
print(alias)

1 no está ('d', 4) {'b': 2, 'c': 3}
{'c': 3}
KeyError: 'z'
KeyError: 'z'
KeyError: 'popitem(): dictionary is empty'
{'k': 1}
{}


## 12. Comparar y unir diccionarios

¿Qué dan `{'a': 1} < {'a': 2}`, `a | b`, `b | a`, `a | b == b | a` y `a.update(b)` con `a = {'x': 1, 'y': 2}` y `b = {'y': 20, 'z': 30}`?

Los diccionarios solo admiten `==` y `!=`. Al unirlos, en las claves repetidas gana el de la derecha, así que `|` no es conmutativo. `update` modifica `a` y devuelve `None`.

In [14]:
try:
    {"a": 1} < {"a": 2}
except TypeError as e:
    print("TypeError:", e)

a, b = {"x": 1, "y": 2}, {"y": 20, "z": 30}
print(a | b, b | a, a | b == b | a, {**a, **b} == a | b)
resultado = a.update(b)
print(resultado, a)

TypeError: '<' not supported between instances of 'dict' and 'dict'
{'x': 1, 'y': 20, 'z': 30} {'y': 2, 'z': 30, 'x': 1} False True
None {'x': 1, 'y': 20, 'z': 30}


## 13. Claves que cambian

¿Qué pasa si se usa como clave un objeto mutable con `__eq__` y `__hash__` basados en sus atributos, y después se modifica?

El diccionario guardó el *hash* que tenía la clave al insertarla. Tras cambiarla, ni la propia clave ni un objeto igual al valor original la encuentran, aunque sigue dentro del diccionario. Por eso las claves deben ser inmutables, por ejemplo una `dataclass(frozen=True)`.

In [15]:
from dataclasses import dataclass

class Punto:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, otro):
        return (self.x, self.y) == (otro.x, otro.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Punto({self.x}, {self.y})"

p = Punto(1, 2)
d = {p: "tesoro"}
p.x = 5
print(d, p in d, Punto(1, 2) in d, Punto(5, 2) in d, list(d)[0] is p)

@dataclass(frozen=True)
class PuntoFijo:
    x: int
    y: int

q = PuntoFijo(1, 2)
d = {q: "tesoro"}
try:
    q.x = 5
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")
print(d[PuntoFijo(1, 2)])

{Punto(5, 2): 'tesoro'} False False False True
FrozenInstanceError: cannot assign to field 'x'
tesoro


## 14. Variantes de un contador de palabras

Contar las palabras de un texto de cuatro formas: con `if`/`else`, con `get`, con `setdefault` y con `Counter`, y comprobar que dan lo mismo. ¿Qué falla en la versión que hace directamente `cuenta[p] += 1`?

In [16]:
from collections import Counter

palabras = "el perro y el gato y el ratón".split()

v1 = {}
for p in palabras:
    if p in v1:
        v1[p] += 1
    else:
        v1[p] = 1

v2 = {}
for p in palabras:
    v2[p] = v2.get(p, 0) + 1

v3 = {}
for p in palabras:
    v3.setdefault(p, 0)
    v3[p] += 1

v4 = Counter(palabras)
assert v1 == v2 == v3 == v4
print(v1, v4.most_common(2))

cuenta = {}
try:
    for p in palabras:
        cuenta[p] += 1
except KeyError as e:
    print("KeyError:", e)

{'el': 3, 'perro': 1, 'y': 2, 'gato': 1, 'ratón': 1} [('el', 3), ('y', 2)]
KeyError: 'el'


## 15. Lista de diccionarios y diccionario de listas

Convertir una lista de diccionarios (filas de una tabla) en un diccionario de listas (columnas) y al revés, comprobando que el viaje de ida y vuelta devuelve los datos originales. ¿Qué pasa si a alguna fila le falta un campo?

In [17]:
filas = [{"nombre": "Ana", "edad": 31}, {"nombre": "Luis", "edad": 26}, {"nombre": "Eva", "edad": 34}]
columnas = {k: [f[k] for f in filas] for k in filas[0]}
print(columnas)
vuelta = [dict(zip(columnas, valores)) for valores in zip(*columnas.values())]
assert vuelta == filas

incompletas = filas + [{"nombre": "Raúl"}]
try:
    {k: [f[k] for f in incompletas] for k in incompletas[0]}
except KeyError as e:
    print("KeyError:", e)
print({k: [f.get(k) for f in incompletas] for k in incompletas[0]})

{'nombre': ['Ana', 'Luis', 'Eva'], 'edad': [31, 26, 34]}
KeyError: 'edad'
{'nombre': ['Ana', 'Luis', 'Eva', 'Raúl'], 'edad': [31, 26, 34, None]}
